# ScreenWise — AI Component Demonstration

**ScreenWise – AI-Based Intelligent CV Screening and Candidate Ranking System for IT Recruitment**

This notebook runs the **actual Python AI service code** of the project (`ai-service/app`) and the
**actual IT datasets and Model 2 artefacts** in the repository. Nothing here is a re-implementation or a mock.

## 1. Project introduction

**The problem.** An IT recruiter receives hundreds of CVs (PDF / DOCX / scanned images) for one vacancy. Reading them
all is slow and inconsistent, and keyword search misses related skills (e.g. *FastAPI* vs *Python web framework*).

**What the AI does.** The AI service performs two different jobs:

| | **CV information extraction** | **Candidate ranking** |
|---|---|---|
| Input | one raw CV file (PDF / DOCX / image) | a job description **plus** several already-extracted candidate profiles |
| Output | a structured `CandidateProfile` (name, email, skills, education, experience, projects, …) | an ordered list of candidates with a match % and an explanation |
| Technique | OCR / text parsing → text cleaning → spaCy NER (if the trained model file is present) + rule-based section parser | weighted multi-criteria scoring (skills, experience, education, projects, communication, certifications) fused with sentence-embedding similarity (`all-MiniLM-L6-v2`); optional LightGBM blend if a trained `.pkl` is present |
| Question answered | *"What does this CV say?"* | *"Which of these candidates fits this job best?"* |

Extraction feeds ranking: the profile produced by extraction is exactly the candidate input that ranking consumes.

**How it fits the application.** React frontend → Node.js/Express backend (MySQL) → this FastAPI service
(`/api/v1/extract`, `/api/v1/rank`, …). The backend calls the AI service when HR uploads a CV or clicks *Rank*.

> **Honesty notes used throughout this notebook**
> * Every printed number is computed when the cell runs, or is explicitly labelled as a *stored* value read from a file.
> * The datasets are **synthetic** (generated for the research project). Metrics therefore measure agreement with the
>   dataset's generation rubric, **not** real-world hiring accuracy.
> * The 'Model 2' LightGBM LambdaMART ranker in `model2_it/` is a research artefact; the live API ranking path
>   is the weighted + embedding pipeline (see Section 5 and 6).

## 2. Environment setup (Windows)

Run these in **Windows Command Prompt / PowerShell** from the project root (the folder that contains `ai-service`, `backend`, `frontend`):

```bat
cd ai-service
python -m venv .venv
.venv\Scripts\activate
python -m pip install --upgrade pip
pip install -r requirements.txt
pip install jupyter ipykernel
python -m ipykernel install --user --name screenwise --display-name "ScreenWise (ai-service)"
jupyter notebook notebooks\ScreenWise_AI_Demonstration.ipynb
```

PowerShell users: activate with `.venv\Scripts\Activate.ps1` (run `Set-ExecutionPolicy -Scope Process RemoteSigned` first if scripts are blocked).

Notes
* Use **Python 3.10 or newer** — the service code uses `X | None` type syntax.
* `requirements.txt` already contains `torch`, `sentence-transformers`, `spacy`, `lightgbm`; do **not** install them a second time.
* The first run downloads the `all-MiniLM-L6-v2` embedding model (~90 MB) from Hugging Face unless it is already cached. This is the only network access.
* Docker is **not** required. Select the kernel **"ScreenWise (ai-service)"** (or the Python from `.venv`) in Jupyter.
* Do not paste API keys into the notebook. The setup cell below blanks the Gemini and Pinecone keys for this session so no cloud service is called.

In [1]:
import os, sys, json, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# --- locate the repository root without hard-coded machine paths -----------------
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "ai-service" / "app" / "main.py").exists():
            return p
    raise RuntimeError("Run this notebook from inside the ScreenWise project folder.")

REPO = find_repo_root(Path.cwd().resolve())
AI = REPO / "ai-service"
M2 = REPO / "model2_it"

# The service resolves relative model paths (e.g. models/extraction/ner_model) from its working directory.
os.chdir(AI)
for p in (str(AI), str(M2)):
    if p not in sys.path:
        sys.path.insert(0, p)

# No cloud calls from this notebook: blank optional API keys for this process only (nothing is written to .env).
os.environ["GEMINI_API_KEY"] = ""
os.environ["PINECONE_API_KEY"] = ""

print("Python      :", platform.python_version(), "|", platform.system())
print("Repo root   :", REPO.name, "(path resolved automatically)")
for pkg in ("fastapi", "pydantic", "spacy", "sentence_transformers", "lightgbm", "sklearn", "pandas", "numpy"):
    try:
        mod = __import__(pkg)
        print(f"{pkg:22s}", getattr(mod, "__version__", "?"))
    except Exception as exc:
        print(f"{pkg:22s} NOT AVAILABLE ->", exc)

artefacts = {
    "spaCy NER model (Model 1)       ": AI / "models/extraction/ner_model",
    "LightGBM blend .pkl (live API)  ": AI / "models/ranking/lgbm_ranker.pkl",
    "Ranking calibration.json        ": AI / "models/ranking/calibration.json",
    "Model 2 LambdaMART booster      ": M2 / "models/best_model/lgbm_ranker.txt",
    "Model 2 test features            ": M2 / "features/test_features.csv",
    "IT ranking CSVs (processed)     ": AI / "datasets/processed/ranking",
    "IT NER JSON (processed)         ": AI / "datasets/processed/ner",
}
print()
for label, path in artefacts.items():
    print(f"{label}: {'FOUND  ' if path.exists() else 'MISSING'}")

Python      : 3.11.17 | Darwin
Repo root   : screenwise_ai_cv_screening (path resolved automatically)


fastapi                0.143.1
pydantic               2.14.1


spacy                  3.8.16


sentence_transformers  6.1.0
lightgbm               4.7.0
sklearn                1.9.1
pandas                 3.0.6
numpy                  2.4.6

spaCy NER model (Model 1)       : MISSING
LightGBM blend .pkl (live API)  : MISSING
Ranking calibration.json        : FOUND  
Model 2 LambdaMART booster      : FOUND  
Model 2 test features            : FOUND  
IT ranking CSVs (processed)     : FOUND  
IT NER JSON (processed)         : FOUND  


## 3. Dataset inspection

Two **synthetic IT datasets** are used (cleaned copies live in `ai-service/datasets/processed/`; originals are never modified):

* **NER dataset (Model 1)** — annotated CV texts for training the entity recogniser. Labels: `NAME, EMAIL, SKILL, DEGREE, CERTIFICATION`.
* **Ranking dataset (Model 2 inputs)** — candidate profiles with dimension scores, IT job descriptions, and labelled candidate–job pairs.

Columns that look like personal data (`name`, `email`, `phone`) are **dropped before display**.

In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", 70, "display.width", 160)

RANK_DIR = AI / "datasets/processed/ranking"
NER_DIR = AI / "datasets/processed/ner"
PII = ["name", "email", "phone"]

cand = {s: pd.read_csv(RANK_DIR / f"it_{s}.csv") for s in ("train", "val", "test")}
cand_full = pd.read_csv(RANK_DIR / "it_candidates_full.csv")
jobs = pd.read_csv(RANK_DIR / "it_job_descriptions.csv")
pairs = pd.read_csv(RANK_DIR / "it_candidate_job_matches.csv")

print("Candidate splits (rows):", {k: len(v) for k, v in cand.items()}, "| full:", len(cand_full))
print("Jobs:", len(jobs), "| labelled candidate-job pairs:", len(pairs))
print()
print("Overlap of candidate_id between splits (must be 0 to avoid leakage):",
      {"train&val": len(set(cand['train'].candidate_id) & set(cand['val'].candidate_id)),
       "train&test": len(set(cand['train'].candidate_id) & set(cand['test'].candidate_id)),
       "val&test": len(set(cand['val'].candidate_id) & set(cand['test'].candidate_id))})
print()
print("Candidate shortlist label (1 = shortlist, 0 = reject), full set:")
print(cand_full["label"].value_counts().rename({1: "shortlist", 0: "reject"}).to_string())
print()
print("Pair relevance_label (1 = relevant):")
print(pairs["relevance_label"].value_counts().to_string())

Candidate splits (rows): {'train': 1050, 'val': 225, 'test': 225} | full: 1497
Jobs: 200 | labelled candidate-job pairs: 2000

Overlap of candidate_id between splits (must be 0 to avoid leakage): {'train&val': 0, 'train&test': 0, 'val&test': 0}

Candidate shortlist label (1 = shortlist, 0 = reject), full set:
label
reject       1117
shortlist     380

Pair relevance_label (1 = relevant):
relevance_label
0    1859
1     141


In [3]:
col_meaning = {
    "candidate_id": "unique synthetic id",
    "years_experience / experience_months": "total experience",
    "technical_skills / soft_skills": "'|' separated skill lists",
    "education_degree / education_classification": "degree and class",
    "projects / certifications": "'|' separated names",
    "score_skills ... score_certifications": "per-criterion score 0-100 produced by the dataset generator",
    "overall_score": "weighted sum of the criterion scores (IT formula)",
    "label": "1 if overall_score >= 60 (shortlist) else 0",
    "candidate_tier": "high / medium / low",
}
print(pd.Series(col_meaning).to_string())
print()
cand_full.drop(columns=PII).head(3).T

candidate_id                                                                           unique synthetic id
years_experience / experience_months                                                      total experience
technical_skills / soft_skills                                                   '|' separated skill lists
education_degree / education_classification                                               degree and class
projects / certifications                                                              '|' separated names
score_skills ... score_certifications          per-criterion score 0-100 produced by the dataset generator
overall_score                                            weighted sum of the criterion scores (IT formula)
label                                                          1 if overall_score >= 60 (shortlist) else 0
candidate_tier                                                                         high / medium / low



,0,1,2
candidate_id,IT-0001,IT-0002,IT-0003
years_experience,8.2,5.5,0.7
experience_months,98,66,8
technical_skills,Pandas|OWASP|React Native|Java|Redis|Android|Express.js|iOS|Firebase,Deep Learning|PHP|Python|FastAPI|DynamoDB|Spring Boot|CSS3|Postgre...,Svelte
soft_skills,Leadership|Communication|Problem Solving|Analytical Thinking,Collaboration|Adaptability,Analytical Thinking|Teamwork
num_technical_skills,9,11,1
num_soft_skills,4,2,2
education_degree,MBA Information Technology,BSc Information Technology,MSc Software Engineering
education_classification,Lower Second Class (2:2),First Class Honours,Merit
num_projects,7,7,2


In [4]:
print("IT job descriptions — columns:", list(jobs.columns))
display(jobs.head(3))
print("Candidate-job pair columns:", list(pairs.columns))
display(pairs.head(3).drop(columns=[]))

IT job descriptions — columns: ['job_id', 'title', 'industry', 'required_skills', 'preferred_skills', 'min_experience_years', 'description']


,job_id,title,industry,required_skills,preferred_skills,min_experience_years,description
0,JOB-IT-001,Cloud Architect,IT,JavaScript|Python|PyTorch|Django|C++|Java,Linux|TypeScript|JavaScript,2,"Seeking a Cloud Architect with expertise in JavaScript, Python, Py..."
1,JOB-IT-002,Backend Developer,IT,PostgreSQL|PyTorch|Docker|Java|TensorFlow|Agile|AWS,JavaScript|PyTorch|CI/CD|C++,3,"Seeking a Backend Developer with expertise in PostgreSQL, PyTorch,..."
2,JOB-IT-003,Software Engineer,IT,MySQL|C++|Node.js|Docker|Git|Kubernetes,PostgreSQL|React.js|C++,2,"Seeking a Software Engineer with expertise in MySQL, C++, Node.js...."


Candidate-job pair columns: ['candidate_id', 'job_id', 'job_title', 'cv_skills', 'job_required_skills', 'cv_overall_score', 'years_experience', 'tfidf_similarity', 'relevance_label']


,candidate_id,job_id,job_title,cv_skills,job_required_skills,cv_overall_score,years_experience,tfidf_similarity,relevance_label
0,IT-0022,JOB-IT-161,Technical Lead,C++|scikit-learn,Node.js|Git|React.js|TypeScript|Redis|Flask|Angular,26,0.6,0.0000,0
1,IT-0955,JOB-IT-133,Software Architect,Flutter|Vue.js|Machine Learning,Kubernetes|Flask|FastAPI|scikit-learn|Node.js,31,1.0,0.0935,0
2,IT-1129,JOB-IT-081,QA Engineer,Spring Boot|DevOps|Azure|Google Cloud|Webpack|TDD|Jest|Express.js|...,React.js|Flask|scikit-learn|JavaScript|PyTorch|MongoDB|CI/CD,76,3.8,0.0773,0


In [5]:
ner = {s: json.loads((NER_DIR / f"it_ner_{s}.json").read_text(encoding="utf-8")) for s in ("train", "val", "test")}
print("NER records per split:", {k: len(v) for k, v in ner.items()})

from collections import Counter
for split, data in ner.items():
    c = Counter(lab for rec in data for _, _, lab in rec["entities"])
    print(f"{split:5s} entity counts:", dict(c))

# Show the annotation format WITHOUT contact details: only skill / degree / certification spans.
rec = ner["test"][0]
safe = [(rec["text"][s:e], lab) for s, e, lab in rec["entities"] if lab in ("SKILL", "DEGREE", "CERTIFICATION")]
print()
print("Annotation format: record = {'text': str, 'entities': [[start, end, LABEL], ...]}")
print("Example (NAME/EMAIL spans hidden):", safe[:8])
overlap = len({r['text'] for r in ner['train']} & {r['text'] for r in ner['test']})
print("Identical CV texts shared by train and test:", overlap)

NER records per split: {'train': 700, 'val': 150, 'test': 150}
train entity counts: {'NAME': 697, 'EMAIL': 700, 'DEGREE': 700, 'SKILL': 4398, 'CERTIFICATION': 908}
val   entity counts: {'NAME': 150, 'EMAIL': 150, 'DEGREE': 150, 'SKILL': 966, 'CERTIFICATION': 196}
test  entity counts: {'NAME': 150, 'EMAIL': 150, 'DEGREE': 150, 'SKILL': 985, 'CERTIFICATION': 205}

Annotation format: record = {'text': str, 'entities': [[start, end, LABEL], ...]}
Example (NAME/EMAIL spans hidden): [('BSc Computer Engineering', 'DEGREE'), ('Redis', 'SKILL'), ('Pandas', 'SKILL'), ('JWT', 'SKILL'), ('Cybersecurity', 'SKILL'), ('CI/CD', 'SKILL'), ('Microservices', 'SKILL'), ('Go', 'SKILL')]
Identical CV texts shared by train and test: 0


## 4. Model 1 — CV information extraction (existing implementation)

`ExtractionService` (`ai-service/app/services/extraction_service.py`) runs:

1. `OCRService` — PDF via PyMuPDF, DOCX via python-docx, images via Tesseract (Gemini fallback is optional and **disabled** here).
2. `clean_text` — normalises whitespace / bullets.
3. `SectionParser` — **rule-based** extraction of skills, education, experience, projects, contact details.
4. `NERRepository` — trained **spaCy NER** model *if the file `models/extraction/ner_model` exists*; otherwise the service logs a warning and continues rule-only.
5. The results are merged into a `CandidateProfile`.

The cell below builds a small **synthetic** DOCX CV in memory (no real person) and passes it through the real service.

In [6]:
import io
from docx import Document
from app.services.extraction_service import ExtractionService

def make_sample_docx() -> bytes:
    d = Document()
    for line in [
        "Nimal Perera",
        "nimal.perera@example.com | +94 77 000 1234",
        "SKILLS", "Python, FastAPI, PostgreSQL, Docker, React, Git",
        "EXPERIENCE", "Software Engineer at Example Tech, 2021 - 2024: built REST APIs and CI/CD pipelines.",
        "EDUCATION", "BSc in Computer Science, Example University, 2020",
        "PROJECTS", "CV Screening Tool - Python, spaCy, FastAPI",
        "CERTIFICATIONS", "AWS Certified Cloud Practitioner",
    ]:
        d.add_paragraph(line)
    buf = io.BytesIO(); d.save(buf); return buf.getvalue()

extractor = ExtractionService()
print("Trained spaCy NER model loaded :", extractor.ner_loaded)
resp = extractor.extract_from_bytes("synthetic_sample_cv.docx", make_sample_docx())

out = resp.model_dump()
out["profile"].pop("raw_text", None)          # keep the output short
print(json.dumps(out, indent=2)[:3500])

Tesseract not available — image OCR disabled: C:\Program Files\Tesseract-OCR\tesseract.exe is not installed or it's not in your PATH. See README file for more information.


NER model not found at models/extraction/ner_model — using rule-only extraction


Trained spaCy NER model loaded : False


{
  "success": true,
  "filename": "synthetic_sample_cv.docx",
  "profile": {
    "name": "Nimal Perera",
    "email": "nimal.perera@example.com",
    "phone": "+94 77 000 1234",
    "address": null,
    "linkedin": null,
    "github": null,
    "portfolio": null,
    "education": [
      {
        "university": "University",
        "degree": "BSc in Computer Science",
        "graduation_year": "2020",
        "field": null
      }
    ],
    "experience": [
      {
        "company": null,
        "job_title": "Software Engineer at Example Tech, 2021 - 2024: built REST APIs and CI/CD pipelines.",
        "duration": "2021 - 2024",
        "description": null
      }
    ],
    "technical_skills": [
      "Python",
      "FastAPI",
      "PostgreSQL",
      "Docker",
      "React",
      "Git"
    ],
    "soft_skills": [],
    "languages": [],
    "projects": [
      {
        "name": "CV Screening Tool - Python, spaCy, FastAPI",
        "description": "CV Screening Tool - Python, sp

**Reading the returned structure** (`ExtractResponse` → `profile: CandidateProfile`)

| Field | Meaning |
|---|---|
| `name`, `email`, `phone`, `linkedin`, `github` | contact fields (regex / NER) |
| `technical_skills`, `soft_skills` | skill lists found by the section parser (plus NER skills when the model is loaded) |
| `education[]` | `degree`, `university`, `graduation_year`, `field` |
| `experience[]` | `job_title`, `company`, `duration`, `description` |
| `projects[]` | project `name`, `description`, detected `technologies` |
| `certifications`, `awards`, `languages` | supporting lists |
| `years_of_experience` | numeric estimate (may be `null` if the CV does not state it) |
| `communication_score` | 0-1 heuristic writing-quality score (`app/utils/communication_score.py`), **not** a NER output |
| `extraction_meta` | how the text was obtained (`method`) and which components ran |

Imperfections are visible in the output above (for example generic university names or an unparsed company). That is the
genuine behaviour of the rule-based parser on this input.

In [7]:
# Optional: the same extraction through the real HTTP API, if the FastAPI service is running on port 8000.
import httpx
try:
    h = httpx.get("http://localhost:8000/api/v1/health", timeout=3).json()
    print("Live AI service health:", h)
    r = httpx.post("http://localhost:8000/api/v1/extract",
                   files={"file": ("synthetic_sample_cv.docx", make_sample_docx(),
                                   "application/vnd.openxmlformats-officedocument.wordprocessingml.document")},
                   timeout=120)
    print("POST /api/v1/extract ->", r.status_code, "| skills:", r.json()["profile"]["technical_skills"])
except Exception as exc:
    print("FastAPI service not reachable on :8000 (that is fine — the cells above call the same code in-process).")
    print("Start it with:  cd ai-service  &&  python -m uvicorn app.main:app --port 8000")

Live AI service health: {'status': 'ok', 'version': '1.0.0', 'ner_model_loaded': False, 'embedding_model_loaded': True, 'vector_backend': 'local'}
POST /api/v1/extract -> 200 | skills: ['Python', 'FastAPI', 'PostgreSQL', 'Docker', 'React', 'Git']


## 5. IT candidate ranking (existing live pipeline)

`RankingService` (`ai-service/app/services/ranking_service.py`) is **not a classifier**. For every candidate it computes:

1. **Rule-based criterion scores** (0-1) for `technical_skills, experience, projects, education, communication, certifications`.
2. These are combined with the **IT weight template** (`GET /api/v1/ranking/weights/it`) into a weighted score.
3. Optionally blended with a trained LightGBM model **only if** `models/ranking/lgbm_ranker.pkl` exists *and* `ml_blend_weight > 0` in `calibration.json`.
4. **Fused with embedding similarity**: `final = (1 - w) * base + w * semantic_similarity` where `semantic_similarity` is the cosine similarity of
   `all-MiniLM-L6-v2` sentence embeddings of the job and the candidate text and `w = semantic_fusion_weight` (from `calibration.json`).

So each value below is one of: **rule-based score** (criteria), **embedding-based similarity** (`semantic_similarity`), or the **fused match %**.

In [8]:
from app.services.ranking_service import RankingService
from app.configs.ranking_weights import get_industry_weights
from app.schemas.models import (CandidateProfile, EducationEntry, JobRequirements, ProjectEntry,
                                RankCandidateInput, RankRequest)

ranker = RankingService()
print("IT criterion weights (rule-based):", get_industry_weights("it"))
print("Embedding model loaded           :", ranker.embedding_loaded, "(loads lazily on first use)")
print("LightGBM blend model loaded      :", ranker.ranker.is_loaded)
print("semantic_fusion_weight           :", ranker.ranker.get_semantic_fusion_weight(0.0))
print("ml_blend_weight in effect        :", ranker.ranker.get_ml_blend_weight(0.0))

IT criterion weights (rule-based): {'technical_skills': 0.35, 'experience': 0.25, 'projects': 0.1, 'education': 0.15, 'communication': 0.1, 'certifications': 0.05}
Embedding model loaded           : False (loads lazily on first use)
LightGBM blend model loaded      : False
semantic_fusion_weight           : 0.08
ml_blend_weight in effect        : 0.0


In [9]:
job = JobRequirements(
    title="Backend Python Developer",
    description="Build and maintain REST APIs with FastAPI and PostgreSQL. Containerise services with Docker and deploy on AWS.",
    required_skills=["Python", "FastAPI", "PostgreSQL", "Docker"],
    minimum_experience=2, education_level="bachelor", industry="it",
)
def make_cand(cid, skills, yrs, degree, comm, projects=(), certs=()):
    return RankCandidateInput(candidate_id=cid, profile=CandidateProfile(
        name=f"Synthetic {cid}", technical_skills=list(skills), years_of_experience=yrs,
        education=[EducationEntry(degree=degree)], communication_score=comm,
        projects=[ProjectEntry(name=p, description=p, technologies=list(skills)[:2]) for p in projects],
        certifications=list(certs)))

request = RankRequest(job=job, candidates=[
    make_cand("A", ["Python","FastAPI","PostgreSQL","Docker","AWS"], 4, "BSc Computer Science", 0.8, ["REST API platform"], ["AWS Certified Developer"]),
    make_cand("B", ["Java","Spring","MySQL"], 3, "BSc Software Engineering", 0.6, ["Inventory system"]),
    make_cand("C", ["Python","Django","SQLite"], 1, "Diploma in IT", 0.5),
    make_cand("D", ["HTML","CSS"], 0.5, "Diploma", 0.5),
])
result = ranker.rank(request)

rows = [{"rank": c.ranking_position, "candidate": c.candidate_id, "match_%": c.matching_percentage,
         "embedding_similarity": c.semantic_similarity,
         **{f"{s.criterion}": round(s.score, 2) for s in c.criterion_scores}} for c in result.top_candidates]
display(pd.DataFrame(rows).set_index("rank"))
print("Weights used:", result.weights_used, "| processing_ms:", result.processing_ms)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,candidate,match_%,embedding_similarity,technical_skills,experience,projects,education,communication,certifications
rank,,,,,,,,,
1,A,77.82,0.8496,0.95,0.72,0.45,0.73,0.8,0.50
2,B,37.25,0.2369,0.06,0.67,0.12,0.74,0.6,0.25
3,C,32.93,0.3268,0.28,0.32,0.11,0.52,0.5,0.25
4,D,19.35,0.0653,0.03,0.17,0.11,0.51,0.5,0.25


Weights used: {'technical_skills': 0.35, 'experience': 0.25, 'projects': 0.1, 'education': 0.15, 'communication': 0.1, 'certifications': 0.05} | processing_ms: 6945.07


In [10]:
for c in result.top_candidates:
    e = c.explanation
    print(f"#{c.ranking_position} candidate {c.candidate_id}  match={c.matching_percentage}%")
    print("   matched skills :", e.matched_skills)
    print("   missing skills :", e.missing_skills)
    print("   strengths      :", e.strengths)
    print("   gaps           :", e.gaps)

#1 candidate A  match=77.82%
   matched skills : ['Python', 'FastAPI', 'PostgreSQL', 'Docker']
   missing skills : []
   strengths      : ['Matched IT skills: Python, FastAPI, PostgreSQL, Docker', 'Strong technical skills (95%)', 'Strong communication (80%)']
   gaps           : ['Gap in projects (45%)']
#2 candidate B  match=37.25%
   matched skills : []
   missing skills : ['Python', 'FastAPI', 'PostgreSQL', 'Docker']
   strengths      : ['Strong education (74%)', 'Strong experience (67%)']
   gaps           : ['Missing skills: Python, FastAPI, PostgreSQL, Docker', 'Gap in technical skills (6%)', 'Gap in projects (12%)', 'Gap in certifications (25%)']
#3 candidate C  match=32.93%
   matched skills : ['Python']
   missing skills : ['FastAPI', 'PostgreSQL', 'Docker']
   strengths      : ['Matched IT skills: Python']
   gaps           : ['Missing skills: FastAPI, PostgreSQL, Docker', 'Gap in projects (11%)', 'Gap in certifications (25%)', 'Gap in technical skills (28%)']
#4 candidate D 

## 6. Model 2 — LightGBM LambdaMART ranker (`model2_it/`)

`model2_it/` contains a **learning-to-rank** model (LightGBM `lambdarank`, grouped by job) trained on a second synthetic dataset
(10,000 job-candidate pairs). Its features are engineered from the *raw* job/candidate fields (skill overlap, experience gap, etc.);
the generator's own match components are listed in the manifest as **excluded leakage columns**.

Important facts that the code confirms:

* The live FastAPI path (`/api/v1/rank`) **does not call this booster**. It only exposes the stored metrics via
  `GET /api/v1/ranking/model2/metrics`. The booster is evaluated separately (below).
* Training used `train_features.csv`; hyper-parameters were chosen on `validation_features.csv`; `test_features.csv` is the held-out test set.

The cell loads the real booster and re-scores the **test split only**, using the project's own metric functions (`model2_it/ranking_metrics.py`).
Nothing is written to disk (the project's `evaluate_model2.py` would overwrite `evaluation/model2_metrics.json`, so it is not run here).

In [11]:
import lightgbm as lgb, numpy as np
from ranking_metrics import evaluate_ranking

m2_cfg = json.loads((M2 / "models/best_model/best_model_config.json").read_text())
manifest = json.loads((M2 / "features/feature_manifest.json").read_text())
feature_cols = manifest["feature_columns"]
print("model_type:", m2_cfg["model_type"], "| features:", len(feature_cols), "| hyperparameters:", m2_cfg["hyperparameters"])
print("Leakage columns excluded from features:", manifest["excluded_leakage_columns"])

booster = lgb.Booster(model_file=str(M2 / "models/best_model/lgbm_ranker.txt"))
test_df = pd.read_csv(M2 / "features/test_features.csv").sort_values(["job_id", "candidate_id"]).reset_index(drop=True)
test_df["ml_score"] = booster.predict(test_df[feature_cols].astype(float).values)
test_df["baseline_score"] = test_df["required_skill_overlap"].astype(float)   # skill-overlap-only baseline

ml = evaluate_ranking(test_df, "ml_score")
bl = evaluate_ranking(test_df, "baseline_score")
stored = json.loads((M2 / "evaluation/model2_metrics.json").read_text())

cmp = pd.DataFrame({"recomputed raw LambdaMART": ml, "stored (ml_model_raw)": stored["ml_model_raw"], "recomputed skill-overlap baseline": bl})
print(f"Held-out test: {len(test_df)} pairs, {test_df.job_id.nunique()} jobs")
display(cmp)

imp = pd.Series(booster.feature_importance("gain"), index=feature_cols).sort_values(ascending=False).head(8)
print("Top features by gain (the model's own explanation of what drives scores):")
print((imp / imp.sum()).round(3).to_string())

model_type: lightgbm_lambdarank | features: 413 | hyperparameters: {'num_leaves': 63, 'learning_rate': 0.15, 'n_estimators': 200, 'min_child_samples': 20, 'best_iteration': 142}
Leakage columns excluded from features: ['certification_match', 'education_match', 'experience_match', 'preferred_skill_match', 'project_relevance', 'required_skill_match', 'semantic_similarity']


Held-out test: 1416 pairs, 101 jobs


,recomputed raw LambdaMART,stored (ml_model_raw),recomputed skill-overlap baseline
ndcg_at_5,0.9997,0.9997,0.9759
ndcg_at_10,0.9999,0.9999,0.9846
mrr,1.0000,1.0000,1.0000
precision_at_5,0.8653,0.8653,0.8297
recall_at_5,0.8023,0.8023,0.7653
top1_accuracy,0.9604,0.9604,0.4455
spearman,0.9541,0.9541,0.9032
kendall_tau,0.8456,0.8456,0.7973


Top features by gain (the model's own explanation of what drives scores):
rule_based_composite      0.980
emb_191                   0.007
required_skill_overlap    0.005
emb_213                   0.002
emb_327                   0.002
emb_203                   0.002
emb_259                   0.002
emb_202                   0.002


**How to read these numbers.** NDCG@5 / MRR close to 1.0 mean the model orders candidates almost exactly as the dataset's **generation rubric**
orders them. The project's own report (`model2_it/evaluation/model2_evaluation_report.md`) states the same limitation:
the ground truth is a fixed synthetic rubric, not observed hiring decisions. This is evidence that the pipeline learns the rubric, **not** that it will reach this
accuracy on real recruiters' decisions. `top1_accuracy` is the share of jobs where the model's #1 candidate equals the rubric's #1 candidate — it is not classification accuracy.

**Caution about the near-perfect scores.** The feature-importance table above is computed from the real booster: roughly 98% of the total gain is on a single
feature, `rule_based_composite`, a hand-built weighted sum of the engineered match features (skill overlap, experience, projects, education, certifications, skill-graph
overlap; see `model2_it/prepare_model2_features.py`). The ML model therefore mostly refines an input that is itself a rule-based score, rather than finding an independent signal,
and its margin over the skill-overlap baseline should be quoted with that caveat. This notebook does not claim leakage, but because the labels are synthetic and
rule-generated, a high score here shows consistency with that rule-based construction only.

## 7. Evaluation

### Training / validation / test — and leakage
* **Training** data fits model parameters; **validation** data is used to choose hyper-parameters and blend weights; **test** data is touched only for the final report.
* Model 1 (NER): 700 / 150 / 150 CVs (identical texts across splits: 0, checked in Section 3).
* Ranking candidates: 1050 / 225 / 225 with 0 shared `candidate_id`s (checked in Section 3).
* Model 2 LambdaMART: train / validation / test feature files; metrics in Section 6 are on the test file only.

Below, the **live pipeline** is evaluated on **test-split items only**. The live ranker is rule-based + embeddings (no weights were fitted on these data),
so these evaluations cannot leak training information, and we still restrict to the test split for consistency.

In [12]:
# ---- 7.1 Extraction vs gold annotations on the held-out NER test split (150 CVs) ----
from collections import Counter
from app.repositories.ner_repository import NERRepository

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return round(p, 4), round(r, 4), round(f, 4)

svc = ExtractionService()
tp = fp = fn = 0; email_ok = name_ok = 0
fp_counter = Counter(); fn_counter = Counter()
for rec in ner["test"]:
    text, gold = rec["text"], rec["entities"]
    gold_sk = {text[s:e].strip().lower() for s, e, lab in gold if lab == "SKILL"}
    gold_email = next((text[s:e] for s, e, lab in gold if lab == "EMAIL"), None)
    gold_name = next((text[s:e] for s, e, lab in gold if lab == "NAME"), None)
    prof = svc.extract_from_text(text)
    pred_sk = {x.strip().lower() for x in prof.technical_skills}
    tp += len(gold_sk & pred_sk); fp += len(pred_sk - gold_sk); fn += len(gold_sk - pred_sk)
    fp_counter.update(pred_sk - gold_sk); fn_counter.update(gold_sk - pred_sk)
    email_ok += int((prof.email or "").lower() == (gold_email or "").lower())
    name_ok += int((prof.name or "").strip().lower() == (gold_name or "").strip().lower())

n = len(ner["test"])
p, r, f = prf(tp, fp, fn)
print(f"Extraction pipeline (trained NER loaded: {svc.ner_loaded}) on {n} held-out test CVs")
print(f"  SKILL set-level  precision={p}  recall={r}  F1={f}   (micro-averaged over CVs)")
print(f"  EMAIL exact match = {email_ok}/{n} = {email_ok/n:.3f}")
print(f"  NAME  exact match = {name_ok}/{n} = {name_ok/n:.3f}")
print("  Most frequent false-positive skills:", fp_counter.most_common(8))
print("  Most frequent missed skills        :", fn_counter.most_common(5))

if svc.ner_loaded:
    stp = sfp = sfn = 0
    for rec in ner["test"]:
        gold = {(s, e, lab) for s, e, lab in rec["entities"]}
        pred = {(s, e, lab) for s, e, lab, _ in svc.ner.predict_spans(rec["text"])}
        stp += len(gold & pred); sfp += len(pred - gold); sfn += len(gold - pred)
    print("  Trained spaCy NER, exact-span entity-level P/R/F1:", prf(stp, sfp, sfn))
else:
    print()
    print("The trained spaCy NER model folder is not present in this checkout (it is git-ignored), so the figures above are for the")
    print("rule-based parser only. Metrics STORED at training time (not recomputed here) are in models/extraction/metrics.json:")
    stored_ner = json.loads((AI / "models/extraction/metrics.json").read_text())
    print({k: stored_ner[k] for k in ("dataset", "eval_split", "train_size", "val_size", "test_size") if k in stored_ner})
    for key in ("baseline_metrics", "val_metrics", "test_metrics"):
        if isinstance(stored_ner.get(key), dict):
            print(f"  stored {key}:", {m: stored_ner[key].get(m) for m in ("precision", "recall", "f1")})

Tesseract not available — image OCR disabled: C:\Program Files\Tesseract-OCR\tesseract.exe is not installed or it's not in your PATH. See README file for more information.


NER model not found at models/extraction/ner_model — using rule-only extraction


Extraction pipeline (trained NER loaded: False) on 150 held-out test CVs
  SKILL set-level  precision=0.4731  recall=0.9645  F1=0.6348   (micro-averaged over CVs)
  EMAIL exact match = 150/150 = 1.000
  NAME  exact match = 150/150 = 1.000
  Most frequent false-positive skills: [('soft skills', 150), ('analytical thinking', 55), ('project management', 51), ('january 2024 to december 2026', 17), ('january 2019 to december 2022', 16), ('january 2019 to december 2020', 16), ('january 2025 to december 2026', 16), ('cd', 12)]
  Most frequent missed skills        : [('c#', 15), ('ci/cd', 12), ('r', 8)]

The trained spaCy NER model folder is not present in this checkout (it is git-ignored), so the figures above are for the
rule-based parser only. Metrics STORED at training time (not recomputed here) are in models/extraction/metrics.json:
{'dataset': 'ScreenWise_IT_NER', 'eval_split': 'test', 'train_size': 700, 'val_size': 150, 'test_size': 150}
  stored baseline_metrics: {'precision': 0.9756, 

In [13]:
# ---- 7.2 Live ranking pipeline vs labelled candidate-job pairs (test-split candidates only) ----
from sklearn.metrics import roc_auc_score, average_precision_score
from ranking_metrics import ndcg_at_k, mrr, correlation_metrics

test_ids = set(cand["test"].candidate_id)
tp_pairs = pairs[pairs.candidate_id.isin(test_ids)].copy()
cand_idx = cand["test"].set_index("candidate_id")
job_idx = jobs.set_index("job_id")
split = lambda s: [x for x in str(s).split("|") if x and x != "nan"]

def to_profile(row):
    return CandidateProfile(
        technical_skills=split(row.technical_skills), soft_skills=split(row.soft_skills),
        years_of_experience=float(row.years_experience),
        education=[EducationEntry(degree=row.education_degree)],
        projects=[ProjectEntry(name=p, description=p) for p in split(row.projects)],
        certifications=split(row.certifications), communication_score=float(row.score_communication) / 100.0)

scored = []
for job_id, g in tp_pairs.groupby("job_id"):
    j = job_idx.loc[job_id]
    jr = JobRequirements(title=j.title, description=j.description, required_skills=split(j.required_skills),
                         minimum_experience=float(j.min_experience_years), industry="it")
    cands = [RankCandidateInput(candidate_id=cid, profile=to_profile(cand_idx.loc[cid])) for cid in g.candidate_id]
    res = ranker.rank(RankRequest(job=jr, candidates=cands))
    sc = {c.candidate_id: c.matching_percentage for c in res.top_candidates}
    for cid, lab in zip(g.candidate_id, g.relevance_label):
        scored.append({"job_id": job_id, "candidate_id": cid, "label": int(lab), "score": sc[cid]})
S = pd.DataFrame(scored)
print(f"Evaluated {len(S)} labelled pairs from test-split candidates across {S.job_id.nunique()} jobs; positives = {S.label.sum()}")

if S.label.nunique() == 2:
    print("ROC-AUC (match % vs relevance_label):", round(roc_auc_score(S.label, S.score), 4))
    print("Average precision (PR-AUC)           :", round(average_precision_score(S.label, S.score), 4),
          "| positive rate (random-ranker PR-AUC) =", round(S.label.mean(), 4))
groups = [(g.label.tolist(), g.score.tolist()) for _, g in S.groupby("job_id") if len(g) >= 2 and g.label.sum() > 0]
if groups:
    rels, scs = zip(*groups)
    print(f"NDCG@5 over {len(groups)} jobs that have >=2 candidates and >=1 relevant pair:",
          round(float(np.mean([ndcg_at_k(a, b, 5) for a, b in groups])), 4), "| MRR:", round(mrr(list(rels), list(scs)), 4))
else:
    print("Not enough jobs with >=2 candidates and a positive label to compute NDCG/MRR on this split — not reported.")

Evaluated 325 labelled pairs from test-split candidates across 161 jobs; positives = 28
ROC-AUC (match % vs relevance_label): 0.8939
Average precision (PR-AUC)           : 0.4431 | positive rate (random-ranker PR-AUC) = 0.0862
NDCG@5 over 19 jobs that have >=2 candidates and >=1 relevant pair: 0.9783 | MRR: 0.9737


**What the extraction numbers show.** The rule-based parser recovers almost every gold skill (high recall) but its precision is low on this CV layout.
The false-positive list above shows why: the parser's section-heading table (`SECTION_HEADERS` in `app/services/section_parser.py`) has no entry for the headings
`SOFT SKILLS` or `WORK EXPERIENCE`, so the text under them (soft-skill words, job dates) is read as part of the technical-skills block, and `CI/CD` is split on `/`.
This is a genuine limitation of the rule-based path and is reported here rather than hidden; the trained spaCy NER (when its model folder is present) is the component
designed to improve skill precision, and its stored metrics are in `models/extraction/metrics.json`. (The stored values are printed with the key names used in that file;
the project documentation `docs/IT_DATASET_AND_MODELS.md` reports the 0.9756 precision as the held-out test result. They were produced at training time and are **not** recomputed here.)

**Interpretation limits (please read)**

* `relevance_label` is a synthetic label from the dataset generator; metrics above show agreement with it, not hiring accuracy.
* The pair set contains few positive pairs and few candidates per job, so the NDCG/MRR values rest on **small samples** and should be quoted with that caveat.
* A ranking score (`match_%`) is **not** a classification accuracy. Precision/Recall/F1 are used above only for the *extraction* task
  (skills as a set), and ROC-AUC / PR-AUC / NDCG / MRR / Spearman for *ranking*.
* If a metric cannot be computed because labels or artefacts are missing, the cell prints that instead of a number.

## 8. Conclusion

**What was demonstrated (by executing the project's own code):**
1. The datasets' real schemas, split sizes, and a leakage check (no shared ids / texts across splits).
2. `ExtractionService` turning a CV file into a structured `CandidateProfile`, and its scores against gold annotations on the held-out NER test split.
3. `RankingService` ordering candidates for an IT job using weighted rule-based criteria fused with sentence-embedding similarity, with per-candidate explanations.
4. The Model 2 LightGBM LambdaMART artefact re-scored on its held-out test file, reproducing the stored metrics.

**What the models actually are:** extraction = OCR + rules + (optional) spaCy NER; live ranking = weighted rules + embedding similarity
(+ optional LightGBM blend only if its `.pkl` exists); Model 2 = a separately evaluated learning-to-rank research model.

**Limitations:** synthetic data and synthetic labels; near-perfect Model 2 metrics reflect agreement with the generation rubric;
the trained NER weights and the live LightGBM `.pkl` are git-ignored, so on a fresh checkout extraction is rule-only and ranking has no ML blend unless
those files are copied in; the first run needs network access once to fetch the embedding model; no real recruiter judgements were available.

**Integration:** the React frontend calls the Node.js/Express backend (`/api/...`, JWT auth, MySQL). On CV upload the backend posts the file to this
service's `/api/v1/extract`; on *Rank* it posts the job and stored profiles to `/api/v1/rank` and stores the returned match %, rank and explanation.
Start order for a full demo: MySQL → AI service (port 8000) → backend (port 5000) → frontend (port 5173); see `docs/RUN_LOCALLY_WINDOWS.md`.